In [18]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\reask"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\reask\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\reask\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Reask - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Reask - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Reask - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [19]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                   X              Y     rotation
count    5698.000000    5698.000000  5698.000000
mean   700524.400227  755965.808447    84.850825
std       339.721175     341.895172    89.860465
min    699739.459559  755207.635348     0.000000
25%    700272.279549  755702.234548     0.000000
50%    700535.557819  755947.045513     0.000000
75%    700763.925649  756257.592433   180.000000
max    701190.638318  756686.279054   180.000000
<class 'pandas.DataFrame'>
RangeIndex: 5698 entries, 0 to 5697
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      5698 non-null   str    
 1   layer     5698 non-null   str    
 2   handle    5698 non-null   str    
 3   name      5698 non-null   str    
 4   X         5698 non-null   float64
 5   Y         5698 non-null   float64
 6   rotation  5698 non-null   float64
 7   UFV       5698 non-null   str    
 8   PST       5698 non-null   str    
 9   INV       5688 non-null   str

In [20]:
struct_mask = (ELM_df['name'] == "EMF422-1_string_tilt_15°") | (ELM_df['name'] == "*U59")

struct_df = ELM_df[struct_mask].copy()
struct_df.rename(columns={'X': 'Position_X', 'Y': 'Position_Y'}, inplace=True)
struct_df.drop(columns=['rotation', 'type', 'handle', 'QBT', 'DSJ', 'PVM', 'VBT'], inplace=True)
struct_df.to_excel(r"C:\Users\Usuario\Desktop\teste2.xlsx")
print(struct_df)
print(struct_df.describe())
print(struct_df.info())

               layer                      name     Position_X     Position_Y  \
0     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699739.459559  755526.952790   
1     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699751.179559  755526.952790   
2     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699762.899559  755526.952790   
3     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699774.619559  755526.952790   
4     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699786.339559  755526.952790   
...              ...                       ...            ...            ...   
5571  EMF_em_pst_1.3                      *U59  700800.085649  756259.982531   
5572  EMF_em_pst_1.3                      *U59  700764.925649  756270.482531   
5573  EMF_em_pst_1.3                      *U59  700846.965649  756249.482531   
5574  EMF_em_pst_1.3                      *U59  700635.905649  756336.362605   
5577  EMF_em_pst_2.1                      *U59  700842.529989  755847.852212   

     UFV PST INV EFX STR GRP  
0     03

In [21]:
inverter_mask = (ELM_df['name'] == "EMF422-1_inversor")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer', 'EFX', 'STR'], inplace=True)
inverter_df['UFV'] = inverter_df['UFV'].astype(str).str.zfill(2)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df)

         inverter_x     inverter_y UFV PST INV GRP VBT QBT DSJ PVM
5431  700436.255315  756667.244626  01  01  01                    
5432  700520.237327  756634.244626  01  01  02                    
5444  700570.176645  756612.244626  01  01  03                    
5433  700595.272417  756601.244626  01  01  04                    
5443  700600.366210  756579.244626  01  01  05                    
...             ...            ...  ..  ..  ..  ..  ..  ..  ..  ..
5693  700183.358962  755315.600920  03  03  16                    
5694  700183.358962  755305.100920  03  03  17                    
5695  700180.480602  755284.100920  03  03  18                    
5696  700177.682520  755273.600920  03  03  19                    
5697  700167.317731  755242.100920  03  03  20                    

[200 rows x 10 columns]


In [22]:
struct_df['UFV'] = struct_df["UFV"].astype(str).str.zfill(2)
struct_df['GRP'] = struct_df["UFV"].astype(str) + "." + struct_df["PST"].astype(str) + '.' + struct_df['INV'].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + '.' + inverter_df['INV'].astype('Int64').astype(str).str.zfill(2)

print(struct_df)
print(inverter_df)

               layer                      name     Position_X     Position_Y  \
0     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699739.459559  755526.952790   
1     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699751.179559  755526.952790   
2     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699762.899559  755526.952790   
3     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699774.619559  755526.952790   
4     EMF_em_pst_3.3  EMF422-1_string_tilt_15°  699786.339559  755526.952790   
...              ...                       ...            ...            ...   
5571  EMF_em_pst_1.3                      *U59  700800.085649  756259.982531   
5572  EMF_em_pst_1.3                      *U59  700764.925649  756270.482531   
5573  EMF_em_pst_1.3                      *U59  700846.965649  756249.482531   
5574  EMF_em_pst_1.3                      *U59  700635.905649  756336.362605   
5577  EMF_em_pst_2.1                      *U59  700842.529989  755847.852212   

     UFV PST INV EFX STR       GRP  
0 

In [23]:
struct_df['Comparing X'] = struct_df.groupby('GRP')['Position_X'].transform('mean')

struct_df = struct_df.merge(inverter_df, on='GRP')

struct_df['E or W'] = np.where(
    struct_df['inverter_x'] >= struct_df['Comparing X'],
    'E',
    'W'
)

struct_df.sort_values(by=['UFV_x', 'PST_x', 'INV_x', 'Position_Y', 'Position_X'], ascending=[True, True, True, False, True], inplace=True)

struct_df["STR"] = struct_df.groupby("GRP").cumcount()+1
struct_df["TAG"] = struct_df['GRP'] + "." + struct_df['STR'].astype(str).str.zfill(2)


print(struct_df)
print(struct_df["STR"])

               layer                      name     Position_X     Position_Y  \
1500  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700348.881640  756686.279054   
1501  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700360.601640  756686.279054   
1495  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700349.528977  756675.279054   
1496  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700361.248977  756675.279054   
1497  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700372.968977  756675.279054   
...              ...                       ...            ...            ...   
5437  EMF_em_pst_3.3                      *U59  700145.913727  755228.635348   
5438  EMF_em_pst_3.3                      *U59  700157.633727  755228.635348   
5442  EMF_em_pst_3.3                      *U59  700145.913727  755218.135348   
5443  EMF_em_pst_3.3                      *U59  700157.633727  755218.135348   
5534  EMF_em_pst_3.3                      *U59  700151.773727  755207.635348   

     UFV_x PST_x INV_x EFX  STR       G

In [24]:
def group_structures(df, inverter_position, max_distance=12):
    """
    Groups adjacent structures in trios (3) while respecting:
    - TAG boundaries
    - Maximum distance constraint (12m default)
    - Sorted by Position Y then X
    """
    # Sort by Position X (primary) and Position Y (secondary)

    df = df.sort_values(['Position_Y', 'Position_X'], ascending=[False, True]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['Position_X'].diff(),2)
    df['y_diff'] = round(df['Position_Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. Y is different OR
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        (abs(df['y_diff']) > 0.1) |
        (abs(df['x_diff']) > max_distance) |
        (df.index == 0)
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        if inverter_position == "W":
            group_df = group_df.iloc[::-1].reset_index(drop=True)
        # Split groups larger than 3 into trios
        size = len(group_df)
        if size <= 3:
            result_groups.append(group_df)

            continue

        n_trios = size // 3
        for i in range(n_trios):
            result_groups.append(group_df.iloc[i*3:i*3+3])


        if size % 3:  # Handle groups that have sizes that are not divisible by 3
            if (size % 3) % 2:
                result_groups.append(group_df.iloc[-1:])
            else:
                result_groups.append(group_df.iloc[-2:])

    if inverter_position == "W":
        result_groups = result_groups[::-1]
    df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
    return result_groups

def create_group_positions(groups, inverter_position):
    """Create the output DataFrame from groups"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) == 3:
            if inverter_position == "W":
                first_struct = group_df.iloc[2]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[2]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'TAG-3': third_struct['TAG'],
                'Position_X': second_struct['Position_X'],
                'Position_Y': second_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer'],
                'Layer-3': third_struct['layer']
            })
        elif len(group_df) == 2:
            if inverter_position == "W":
                first_struct = group_df.iloc[1]
                second_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer']
            })
    return pd.DataFrame(group_data)

In [25]:
estruturas_E = struct_df[struct_df['E or W'] == 'E']
estruturas_W = struct_df[struct_df['E or W'] == 'W']

results_E = group_structures(estruturas_E, 'E')
results_W = group_structures(estruturas_W, 'W')
print(results_E)
print(results_W)

group_positions_E = create_group_positions(results_E, "E")
group_positions_W = create_group_positions(results_W, "W")

group_positions = pd.concat([group_positions_E, group_positions_W], ignore_index=True)

# DEBUG: Check what columns actually exist
print("Columns in group_positions after concat:")
print(group_positions.columns.tolist())
print("\nFirst 5 rows:")
print(group_positions.head())
print("\nData types:")
print(group_positions.dtypes)
print("\nUnique Group_Size values:")
print(group_positions['Group_Size'].unique())

group_positions["UFV-1"] = group_positions['TAG-1'].str[0:2]
group_positions['UFV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[0:2],
    None
)
group_positions['UFV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[0:2],
    None
)

group_positions["PST-1"] = group_positions['TAG-1'].str[3:5]
group_positions['PST-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[3:5],
    None
)
group_positions['PST-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[3:5],
    None
)

group_positions["INV-1"] = group_positions['TAG-1'].str[6:8]
group_positions['INV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[6:8],
    None
)
group_positions['INV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[6:8],
    None
)

group_positions["STR-1"] = group_positions['TAG-1'].str[-2:]
group_positions['STR-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[-2:],
    None
)
group_positions['STR-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[-2:],
    None
)

group_positions['GRP-1'] = group_positions['TAG-1'].str[:-3]
group_positions['GRP-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[:-3],
    None
)
group_positions['GRP-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[:-3],
    None
)

print(group_positions)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

[            layer                      name    Position_X     Position_Y  \
0  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700348.88164  756686.279054   
1  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700360.60164  756686.279054   

  UFV_x PST_x INV_x EFX  STR       GRP  ...  QBT  DSJ  PVM E or W  \
0    01    01    01        1  01.01.01  ...                     E   
1    01    01    01        2  01.01.01  ...                     E   

           TAG x_diff y_diff distance group_boundary group_id  
0  01.01.01.01    NaN    NaN      NaN           True        1  
1  01.01.01.02  11.72    0.0    11.72          False        1  

[2 rows x 27 columns],             layer                      name     Position_X     Position_Y  \
2  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700349.528977  756675.279054   
3  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700361.248977  756675.279054   
4  EMF_em_pst_1.1  EMF422-1_string_tilt_15°  700372.968977  756675.279054   

  UFV_x PST_x INV_x EFX  STR       GR

In [26]:
correction_position_x_2str = 5.86
correction_position_y_3str = 2.3204

def generate_script(row):
    group_size = row['Group_Size']

    if group_size == 1:
        return (
            f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")'
        )

    elif group_size == 2:
        if row["Layer-1"] == row["Layer-2"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )
        else:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )

    else:  # group_size == 3
        # Check if all layers are the same
        if row["Layer-1"] == row["Layer-2"] == row["Layer-3"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'
            )
        else:
            script = f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'

            if row["Layer-1"] != row["Layer-2"]:
                script += f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'

            if row["Layer-2"] != row["Layer-3"]:
                script += f'(command-s "._layer" "set" "{row["Layer-3"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'

            return script

# Apply the function to each row
group_positions['Script'] = group_positions.apply(generate_script, axis=1)

print(group_positions)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\Separador de Strings - Reask.scr", 'w') as f:
    f.write('\n'.join(group_positions["Script"].astype(str)) + '\n')

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")

      Group_ID        TAG-1        TAG-2     Position_X     Position_Y  \
0            1  01.01.01.01  01.01.01.02  700348.881640  756686.279054   
1            2  01.01.01.03  01.01.01.04  700361.248977  756675.279054   
2            3  01.01.01.06  01.01.01.07  700384.688977  756675.279054   
3            4  01.01.01.08  01.01.01.09  700361.896315  756664.279054   
4            5  01.01.01.11  01.01.01.12  700397.056315  756664.279054   
...        ...          ...          ...            ...            ...   
2016       972  01.01.08.11  01.01.08.12  700658.945649  756546.362605   
2017       973  01.01.08.05          NaN  700623.785649  756556.862605   
2018       974  01.01.08.06  01.01.08.07  700647.225649  756556.862605   
2019       975  01.01.08.02  01.01.08.03  700635.505649  756567.362605   
2020       976  01.01.08.01          NaN  700623.785649  756577.862605   

      Group_Size         Layer-1         Layer-2        TAG-3         Layer-3  \
0              2  EMF_em_pst_1